In [ ]:
#!pip install requests
#!pip install python-dotenv

In [ ]:
import requests
from dotenv import load_dotenv
import os
load_dotenv()
_cache = {}

def get_trailer(title, year):
    key = f"{title}_{year}"
    if key in _cache:
        return _cache[key]
    try:
        url = "https://www.googleapis.com/youtube/v3/search"
        params = {
            "key": os.getenv("YOUTUBE_KEY"),
            "q": f"{title} {year} official trailer",
            "part": "snippet",
            "maxResults": 1,
            "type": "video"
        }
        r = requests.get(url, params=params)
        r.raise_for_status()
        items = r.json().get("items", [])
        if items:
            video_id = items[0]["id"]["videoId"]
            _cache[key] = f"https://www.youtube.com/watch?v={video_id}"
            return _cache[key]
    except Exception as e:
        print(f"get_trailer error: {e}")
    _cache[key] = None
    return None

In [ ]:
# Step 1 — check key is loaded
print(f"YouTube key: {os.getenv('YOUTUBE_KEY')[:8]}...")

In [ ]:
# Test 1 — well known movie
trailer = get_trailer("Inception", 2010)
print(f"Test 1 — Inception:")
print(f"URL: {trailer}")
print()

In [ ]:
# Test 2 — another movie
trailer = get_trailer("The Godfather", 1972)
print(f"Test 2 — The Godfather:")
print(f"URL: {trailer}")
print()

In [ ]:
# Test 3 — check cache works
import time

start = time.time()
get_trailer("Inception", 2010)  # first call — hits API
first_call = time.time() - start

start = time.time()
get_trailer("Inception", 2010)  # second call — should hit cache
second_call = time.time() - start

print(f"Test 3 — Cache test:")
print(f"First call: {first_call:.2f}s")
print(f"Second call: {second_call:.2f}s")
print(f"Cache working: {second_call < first_call}")

In [ ]:
# Test 4 — verify URL format is correct
trailer = get_trailer("Inception", 2010)
if trailer:
    print(f"Test 4 — URL format correct: {trailer.startswith('https://www.youtube.com/watch?v=')}")
    print(f"URL: {trailer}")
else:
    print("Test 4 — No trailer found")